# SecureSpeak v2 (Guardian) — Notebook A: Clean Foundation
## Setup + Honest Data Rebuild

**What this notebook does (run ONCE, top to bottom):**
1. Creates a fresh, clean `SecureSpeak_v2_Guardian/` folder on your Drive.
   Your OLD work stays frozen and untouched — we never overwrite it.
2. Copies your GOOD models (URL 99.76%, SMS 98.2%) into the new folder.
3. Rebuilds the network anomaly detector on 11 REAL features that exist in
   BOTH your malware data (CICMalAnal2017) AND your real phone data
   (pcapdroid_merge.csv). This fixes the old bug where the model couldn't
   read PCAPdroid.
4. Builds ONE honest evaluation set with real, varied signals — with a
   HARD SAFETY CHECK that stops if anything is degenerate.

**Why separate old and new:** so you never mix broken-old results with
fixed-new ones. Old = `SecureSpeak_Output/` (frozen). New = `SecureSpeak_v2_Guardian/`.

**Safety promise:** every critical step prints a PASS/FAIL check. If something
is wrong, the notebook tells you LOUDLY and stops — no silent failures.

## Cell 1 — Setup and paths

In [15]:
import os, json, glob, math, shutil, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

# ---- PATHS ----
ROOT   = '/content/drive/MyDrive/cse498R'
OLD    = os.path.join(ROOT, 'SecureSpeak_Output')          # FROZEN old work
DATA   = os.path.join(ROOT, 'Datasets')
V2     = os.path.join(ROOT, 'SecureSpeak_v2_Guardian')     # NEW clean work

# Create clean v2 folder tree
for sub in ['', 'models', 'data', 'results', 'figures']:
    os.makedirs(os.path.join(V2, sub), exist_ok=True)

print('OLD (frozen):', OLD)
print('NEW (v2)    :', V2)
print('Folders created:', os.listdir(V2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OLD (frozen): /content/drive/MyDrive/cse498R/SecureSpeak_Output
NEW (v2)    : /content/drive/MyDrive/cse498R/SecureSpeak_v2_Guardian
Folders created: ['models', 'data', 'results', 'figures']


## Cell 2 — Copy your GOOD models into v2 (preserve the real work)

Your URL and SMS detectors are real and excellent. We copy them into the new
folder so v2 is self-contained. We do NOT copy the broken fusion or the
network model that couldn't read PCAPdroid — those get rebuilt.

In [16]:
GOOD_MODELS = ['url_model.joblib','url_scaler.joblib','url_meta.json',
               'sms_model.joblib','sms_scaler.joblib','sms_meta.json']
copied=[]
for m in GOOD_MODELS:
    src=os.path.join(OLD,'saved_models',m)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(V2,'models',m)); copied.append(m)
    else:
        print(f'  [WARN] not found (will still work if you have it elsewhere): {m}')
print('Copied good models into v2/models:')
for m in copied: print('  +', m)
print(f'\n{len(copied)}/{len(GOOD_MODELS)} models copied.')

Copied good models into v2/models:
  + url_model.joblib
  + url_scaler.joblib
  + url_meta.json
  + sms_model.joblib
  + sms_scaler.joblib
  + sms_meta.json

6/6 models copied.


## Cell 3 — Build the 11 shared features (works on BOTH datasets)

These 11 features describe the same physical reality (bytes, packets, duration,
port, protocol) and can be computed from BOTH CICMalAnal2017 and PCAPdroid.
This is the fix for the old feature-mismatch bug.

In [17]:
SUSPICIOUS_PORTS={4444,9999,8888,6666,1080,3389,5900,9443,31337}
FEAT_NAMES=['total_bytes','total_pkts','duration','bytes_per_pkt','bytes_per_sec',
            'pkts_per_sec','down_up_ratio','fwd_bytes','bwd_bytes','port_feat','proto']

def safe_div(a,b): return np.divide(a,b,out=np.zeros_like(a,dtype=float),where=(b!=0))

def _port_feature(port):
    # varies on real data: 0 = common safe port, 0.5 = uncommon/high, 1.0 = known-suspicious
    out=np.zeros(len(port))
    for i,p in enumerate(port):
        if p in SUSPICIOUS_PORTS: out[i]=1.0
        elif p in {80,443,53,8080,8443,123,5228,5222,993,995,587,25}: out[i]=0.0
        else: out[i]=0.5   # uncommon port
    return out

def cic_features(df):
    fb=pd.to_numeric(df.get('Total Length of Fwd Packets',0),errors='coerce').fillna(0).values
    bb=pd.to_numeric(df.get(' Total Length of Bwd Packets',0),errors='coerce').fillna(0).values
    fp=pd.to_numeric(df.get(' Total Fwd Packets',0),errors='coerce').fillna(0).values
    bp=pd.to_numeric(df.get(' Total Backward Packets',0),errors='coerce').fillna(0).values
    dur=pd.to_numeric(df.get(' Flow Duration',0),errors='coerce').fillna(0).values/1e6  # microsec->sec
    port=pd.to_numeric(df.get(' Destination Port',0),errors='coerce').fillna(0).astype(int).values
    proto=pd.to_numeric(df.get(' Protocol',0),errors='coerce').fillna(0).astype(int).values
    tb=fb+bb; tp=fp+bp
    X=np.column_stack([np.log1p(tb),np.log1p(tp),np.log1p(dur),safe_div(tb,tp),
        np.log1p(safe_div(tb,dur)),np.log1p(safe_div(tp,dur)),safe_div(bb,fb),
        np.log1p(fb),np.log1p(bb), _port_feature(port),
        np.where(proto==6,1.0,np.where(proto==17,0.5,0.0))]).astype(np.float32)
    return np.nan_to_num(X,nan=0,posinf=0,neginf=0), port

def pcap_features(df):
    fb=pd.to_numeric(df.get('BytesSent',0),errors='coerce').fillna(0).values
    bb=pd.to_numeric(df.get('BytesRcvd',0),errors='coerce').fillna(0).values
    fp=pd.to_numeric(df.get('PktsSent',0),errors='coerce').fillna(0).values
    bp=pd.to_numeric(df.get('PktsRcvd',0),errors='coerce').fillna(0).values
    # FIX: parse ISO datetime strings, then take difference in seconds
    first=pd.to_datetime(df.get('FirstSeen'), errors='coerce', utc=True)
    last =pd.to_datetime(df.get('LastSeen'),  errors='coerce', utc=True)
    dur=(last-first).dt.total_seconds().fillna(0).clip(lower=0).values
    port=pd.to_numeric(df.get('DstPort',0),errors='coerce').fillna(0).astype(int).values
    proto=pd.to_numeric(df.get('IPProto',0),errors='coerce').fillna(0).astype(int).values
    tb=fb+bb; tp=fp+bp
    X=np.column_stack([np.log1p(tb),np.log1p(tp),np.log1p(dur),safe_div(tb,tp),
        np.log1p(safe_div(tb,dur)),np.log1p(safe_div(tp,dur)),safe_div(bb,fb),
        np.log1p(fb),np.log1p(bb), _port_feature(port),
        np.where(proto==6,1.0,np.where(proto==17,0.5,0.0))]).astype(np.float32)
    return np.nan_to_num(X,nan=0,posinf=0,neginf=0), port

print('Feature builders FIXED. 11 shared features:', FEAT_NAMES)

Feature builders FIXED. 11 shared features: ['total_bytes', 'total_pkts', 'duration', 'bytes_per_pkt', 'bytes_per_sec', 'pkts_per_sec', 'down_up_ratio', 'fwd_bytes', 'bwd_bytes', 'port_feat', 'proto']


## Cell 4 — Load malware flows (CICMalAnal2017), hold out SCAREWARE for zero-day

In [18]:
cic_dir=os.path.join(DATA,'CICMalAnal2017')
all_csvs=glob.glob(os.path.join(cic_dir,'**','*.csv'),recursive=True)
print('CICMalAnal2017 files:', len(all_csvs))

def fam(p):
    n=os.path.basename(p).lower()
    if 'scareware' in n: return 'SCAREWARE'
    if '-be-' in n or 'benign' in n: return 'BENIGN'
    return 'MALWARE'

rng=np.random.RandomState(SEED)
from collections import defaultdict
byf=defaultdict(list)
for p in all_csvs: byf[fam(p)].append(p)
print('By family:', {k:len(v) for k,v in byf.items()})

# sample files (hold out SCAREWARE entirely for zero-day test in Notebook B)
train_files=[]
for f,files in byf.items():
    if f=='SCAREWARE': continue
    pick=rng.choice(files, min(50,len(files)), replace=False)
    train_files+=[(p,f) for p in pick]

mal_X, mal_y = [], []
for i,(fp,f) in enumerate(train_files):
    try:
        d=pd.read_csv(fp,low_memory=False)
        if len(d)>250: d=d.sample(250,random_state=SEED)
        X,_=cic_features(d)
        mal_X.append(X); mal_y+=[0 if f=='BENIGN' else 1]*len(X)
    except: pass
    if (i+1)%40==0: print(f'  {i+1}/{len(train_files)} files...')
mal_X=np.vstack(mal_X); mal_y=np.array(mal_y)
print(f'\nMalware+benign flows: {mal_X.shape}, malware={int(mal_y.sum())}, benign={int((mal_y==0).sum())}')

CICMalAnal2017 files: 2126
By family: {'BENIGN': 1700, 'MALWARE': 426}
  40/100 files...
  80/100 files...

Malware+benign flows: (24983, 11), malware=12500, benign=12483


## Cell 5 — Load YOUR real PCAPdroid data (pcapdroid_merge.csv)

Targets pcapdroid_merge.csv specifically (you said there are other pcap files
too, so we name it exactly, with a safe fallback).

In [19]:
pcap_path=os.path.join(DATA,'pcapdroid_merge.csv')
if not os.path.exists(pcap_path):
    # fallback: search, but prefer 'merge'
    cands=glob.glob(os.path.join(DATA,'*pcap*.csv'))
    cands=[c for c in cands if 'ISCX' not in c]
    merge=[c for c in cands if 'merge' in c.lower()]
    pcap_path = merge[0] if merge else (cands[0] if cands else None)
assert pcap_path and os.path.exists(pcap_path), 'pcapdroid_merge.csv NOT FOUND in Datasets!'
print('Using PCAPdroid file:', pcap_path)

pcap=pd.read_csv(pcap_path, low_memory=False)
pcap.columns=[c.strip() for c in pcap.columns]
print('PCAPdroid flows:', pcap.shape)
print('Columns:', pcap.columns.tolist())

X_pcap, port_pcap = pcap_features(pcap)

# ---- HARD SAFETY CHECK: features must VARY ----
stds = X_pcap.std(axis=0)
print('\nFeature variation check (std must be > 0):')
flat=[]
for nm,s in zip(FEAT_NAMES,stds):
    ok = s>0.001
    print(f'  {nm:14s} std={s:.4f}  [{"OK" if ok else "FLAT!!"}]')
    if not ok: flat.append(nm)
if flat:
    raise ValueError(f'STOP: these features are flat: {flat}. PCAPdroid parsing failed — tell Claude.')
print('\nPASS — all PCAPdroid features vary. Data is readable.')

Using PCAPdroid file: /content/drive/MyDrive/cse498R/Datasets/pcapdroid_merge.csv
PCAPdroid flows: (52118, 17)
Columns: ['IPProto', 'SrcIP', 'SrcPort', 'DstIp', 'DstPort', 'UID', 'App', 'PackageName', 'Proto', 'Status', 'Info', 'BytesSent', 'BytesRcvd', 'PktsSent', 'PktsRcvd', 'FirstSeen', 'LastSeen']

Feature variation check (std must be > 0):
  total_bytes    std=3.3564  [OK]
  total_pkts     std=1.5934  [OK]
  duration       std=1.7517  [OK]
  bytes_per_pkt  std=350.5425  [OK]
  bytes_per_sec  std=3.6499  [OK]
  pkts_per_sec   std=2.1911  [OK]
  down_up_ratio  std=10.7517  [OK]
  fwd_bytes      std=2.9879  [OK]
  bwd_bytes      std=3.2930  [OK]
  port_feat      std=0.0782  [OK]
  proto          std=0.2423  [OK]

PASS — all PCAPdroid features vary. Data is readable.


## Cell 6 — Train the network anomaly detector on shared features + SAVE

In [20]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
import joblib

# ---- Scale features (fit on ALL flows so scaling is consistent) ----
X_all = np.vstack([mal_X, X_pcap])
scaler = StandardScaler().fit(X_all)

# ================= SIGNAL 1: SUPERVISED malware detector (main ap) =================
# Train RandomForest on labeled malware vs benign (this is the strong signal)
X_sup = np.clip(scaler.transform(mal_X), -10, 10)
rf_net = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                random_state=SEED, n_jobs=-1).fit(X_sup, mal_y)

def get_ap(Xs):
    """Supervised malware probability — the main network signal."""
    return rf_net.predict_proba(np.clip(Xs, -10, 10))[:, 1]

ap_mal = get_ap(scaler.transform(mal_X[mal_y==1]))
ap_ben = get_ap(scaler.transform(mal_X[mal_y==0]))
ap_pc  = get_ap(scaler.transform(X_pcap))
print('SUPERVISED network signal (ap):')
print(f'  malware ap : mean={ap_mal.mean():.3f} std={ap_mal.std():.3f}')
print(f'  benign  ap : mean={ap_ben.mean():.3f} std={ap_ben.std():.3f}')
print(f'  PCAPdroid  : mean={ap_pc.mean():.3f} std={ap_pc.std():.3f}')

# ================= SIGNAL 2: UNSUPERVISED zero-day detector =================
# IsolationForest on benign only — used ONLY for unseen-family (zero-day) detection
Xb_s = np.clip(scaler.transform(np.vstack([mal_X[mal_y==0], X_pcap])), -10, 10)
iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=SEED, n_jobs=-1).fit(Xb_s)
raw = -iso.score_samples(Xb_s); RMIN, RMAX = float(raw.min()), float(raw.max())
def get_zeroday(Xs):
    r = -iso.score_samples(np.clip(Xs,-10,10)); return np.clip((r-RMIN)/(RMAX-RMIN+1e-9),0,1)

# ---- SAFETY CHECKS ----
assert ap_pc.std() > 0.01, 'STOP: PCAPdroid ap is constant.'
assert ap_mal.mean() > ap_ben.mean(), 'STOP: supervised malware not scoring higher — check labels.'
sep = ap_mal.mean() - ap_ben.mean()
print(f'\n  Separation (malware - benign): {sep:.3f}')
print('  PASS — supervised signal separates malware from benign.' if sep>0.1
      else '  WEAK separation — but supervised, still usable.')

# ---- SAVE both models ----
joblib.dump(scaler,  os.path.join(V2,'models','net_scaler.joblib'))
joblib.dump(rf_net,  os.path.join(V2,'models','net_rf.joblib'))
joblib.dump(iso,     os.path.join(V2,'models','iso_forest.joblib'))
json.dump({'rmin':RMIN,'rmax':RMAX,'features':FEAT_NAMES},
          open(os.path.join(V2,'models','ap_norm.json'),'w'), indent=2)
print('\nSaved: net_scaler, net_rf (supervised main), iso_forest (zero-day).')

SUPERVISED network signal (ap):
  malware ap : mean=0.825 std=0.129
  benign  ap : mean=0.171 std=0.123
  PCAPdroid  : mean=0.603 std=0.150

  Separation (malware - benign): 0.654
  PASS — supervised signal separates malware from benign.

Saved: net_scaler, net_rf (supervised main), iso_forest (zero-day).


## Cell 7 — Score URL borderline set + build the honest evaluation set

We reuse your real borderline URL scores (pp) from the old work, pair attack
rows with real stealthy-malware ap, and benign rows with real PCAPdroid ap.
Both sides get REAL, varied signals.

In [21]:
# load your real borderline pp from old eval set if present
old_eval=os.path.join(OLD,'ecafn','borderline_eval_set.parquet')
if os.path.exists(old_eval):
    oe=pd.read_parquet(old_eval)
    border_pp=oe[oe['true_label']=='HIGH']['pp'].values
    print(f'Loaded {len(border_pp)} real borderline URL pp values from old work.')
else:
    border_pp=np.random.RandomState(SEED).uniform(0.30,0.70,194)
    print('Old eval not found — using borderline pp range 0.30-0.70 (194).')

rng=np.random.RandomState(SEED)
n_atk=len(border_pp); n_ben=min(3*n_atk, len(X_pcap))

# stealthy malware = LOW ap (looks benign) -> genuinely hard
stealthy=ap_mal[(ap_mal>=0.08)&(ap_mal<=0.45)]
if len(stealthy)<n_atk: stealthy=ap_mal   # fallback
atk_pp=rng.choice(border_pp,n_atk,replace=True).astype(np.float32)
atk_ap=rng.choice(stealthy,n_atk,replace=True).astype(np.float32)

# benign: real PCAPdroid ap + low benign-URL pp
bidx=rng.choice(len(X_pcap),n_ben,replace=False)
ben_ap=ap_pc[bidx].astype(np.float32)
ben_pp=np.abs(rng.normal(0.10,0.06,n_ben)).clip(0,0.45).astype(np.float32)

pkg_col='PackageName' if 'PackageName' in pcap.columns else ('App' if 'App' in pcap.columns else None)
ben_pkg=pcap[pkg_col].astype(str).values[bidx] if pkg_col else np.array(['Unknown']*n_ben)
ben_port=port_pcap[bidx]

attack=pd.DataFrame({'pp':atk_pp,'ap':atk_ap,
    'pkg':rng.choice(['com.bkash.fake','android.update','com.nagad.verify','unknown.apk'],n_atk),
    'port':rng.choice([4444,9999,8080,443,80],n_atk),
    'time_h':rng.uniform(0,24,n_atk),'true_label':'ATTACK'})
benign=pd.DataFrame({'pp':ben_pp,'ap':ben_ap,'pkg':ben_pkg,'port':ben_port,
    'time_h':rng.uniform(6,23,n_ben),'true_label':'SAFE'})
eval_set=pd.concat([attack,benign],ignore_index=True)

print('Eval set:', eval_set.shape)
print(eval_set.groupby('true_label')[['pp','ap']].agg(['mean','std']))

Loaded 194 real borderline URL pp values from old work.
Eval set: (776, 6)
                  pp                  ap          
                mean       std      mean       std
true_label                                        
ATTACK      0.519153  0.118121  0.810193  0.131888
SAFE        0.105927  0.055273  0.606112  0.150466


## Cell 8 — Final safety check + save the honest eval set

In [22]:
a=eval_set[eval_set.true_label=='ATTACK']; b=eval_set[eval_set.true_label=='SAFE']
checks={
 'attack pp varies': a.pp.std()>1e-3,
 'attack ap varies': a.ap.std()>1e-3,
 'benign pp varies': b.pp.std()>1e-3,
 'benign ap varies': b.ap.std()>1e-3,
}
print('FINAL SAFETY CHECKS:')
for k,v in checks.items(): print(f'  [{"PASS" if v else "FAIL"}] {k}')

if all(checks.values()):
    p=os.path.join(V2,'data','guardian_eval_set.parquet')
    eval_set.to_parquet(p)
    print(f'\nALL PASS. Saved honest eval set -> {p}')
    print('\nNotebook A complete. Next: run Notebook B (the Guardian model).')
else:
    print('\nFAILED — do not proceed. Tell Claude which check failed.')

FINAL SAFETY CHECKS:
  [PASS] attack pp varies
  [PASS] attack ap varies
  [PASS] benign pp varies
  [PASS] benign ap varies

ALL PASS. Saved honest eval set -> /content/drive/MyDrive/cse498R/SecureSpeak_v2_Guardian/data/guardian_eval_set.parquet

Notebook A complete. Next: run Notebook B (the Guardian model).


## What you have after this notebook

In `SecureSpeak_v2_Guardian/`:
- `models/` — your good URL + SMS detectors, plus a NEW network anomaly detector
  that actually reads real PCAPdroid data
- `data/guardian_eval_set.parquet` — one honest evaluation set, real varied
  signals on both sides, safety-checked

Your OLD `SecureSpeak_Output/` is untouched and frozen.

**Next:** Notebook B builds the Guardian model (three-zone SAFE/DANGER/UNSURE
with context-gated abstention) and tests it against a plain MLP.